<a href="https://colab.research.google.com/github/DumuthuLakshan/explainable-ai-network-config-validation/blob/main/Data_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import random
import ipaddress
import pandas as pd
import numpy as np
from collections import defaultdict

# =====================================================
# 1. BASIC SETTINGS
# =====================================================

random.seed(42)
np.random.seed(42)

# This gives exactly 5000 records
NORMAL_PER_PROFILE = 600
DIRECT_ERROR_PER_TYPE = 240
CONTEXTUAL_ANOMALY_PER_TYPE = 250
CROSS_LINK_ANOMALY_PER_TYPE = 100

# More network contexts are used so that many realistic subnets can be generated
NETWORKS = [
    "HospitalNet", "SchoolNet", "CampusNet", "OfficeNet",
    "BranchNet", "DataCenterNet", "LabNet", "AdminNet",
    "FinanceNet", "ResearchNet", "CloudNet", "BackupNet"
]

PROFILES = {
    "router_to_router": {
        "normal_prefixes": [30],
        "expected_ranges": ["10"],
        "normal_roles": [("Router", "Router")],
        "normal_ip_base": "10"
    },
    "branch_lan": {
        "normal_prefixes": [24, 25],
        "expected_ranges": ["192.168"],
        "normal_roles": [("Router", "PC"), ("Router", "Switch")],
        "normal_ip_base": "192.168"
    },
    "campus_network": {
        "normal_prefixes": [23, 24],
        "expected_ranges": ["172.16"],
        "normal_roles": [("Router", "Switch"), ("Router", "AccessPoint")],
        "normal_ip_base": "172.16"
    },
    "datacenter_segment": {
        "normal_prefixes": [28, 29],
        "expected_ranges": ["10"],
        "normal_roles": [("Firewall", "Server"), ("Switch", "Server")],
        "normal_ip_base": "10"
    }
}

DEVICE_POOLS = {
    "Router": ["R1", "R2", "R3", "R4", "R5"],
    "Switch": ["SW1", "SW2", "SW3", "SW4"],
    "PC": ["PC1", "PC2", "PC3", "PC4", "PC5"],
    "Server": ["SRV1", "SRV2", "SRV3", "SRV4"],
    "Firewall": ["FW1", "FW2"],
    "AccessPoint": ["AP1", "AP2", "AP3"]
}

DIRECT_ERRORS = [
    "duplicate_ip",
    "subnet_mask_mismatch",
    "network_mismatch",
    "status_mismatch",
    "invalid_host_ip"
]

CONTEXTUAL_ANOMALIES = [
    "profile_subnet_mismatch",
    "device_role_mismatch",
    "ip_range_profile_mismatch",
    "gateway_role_mismatch"
]

USED_SUBNETS_BY_NETWORK = defaultdict(list)


# =====================================================
# 2. HELPER FUNCTIONS
# =====================================================

def prefix_to_mask(prefix):
    return str(ipaddress.IPv4Network(f"0.0.0.0/{prefix}").netmask)


def get_prefix(mask):
    return ipaddress.IPv4Network(f"0.0.0.0/{mask}").prefixlen


def get_network_address(ip, mask):
    return str(ipaddress.ip_network(f"{ip}/{mask}", strict=False).network_address)


def get_network_cidr(ip, mask):
    return str(ipaddress.ip_network(f"{ip}/{mask}", strict=False))


def is_valid_host_ip(ip, mask):
    network = ipaddress.ip_network(f"{ip}/{mask}", strict=False)
    ip_obj = ipaddress.ip_address(ip)

    if ip_obj == network.network_address:
        return False

    if ip_obj == network.broadcast_address:
        return False

    return True


def subnet_capacity(prefix):
    if prefix >= 31:
        return 0

    return (2 ** (32 - prefix)) - 2


def get_ip_range_type(ip):
    parts = ip.split(".")
    first = int(parts[0])
    second = int(parts[1])

    if first == 10:
        return "10_private"

    if first == 192 and second == 168:
        return "192_private"

    if first == 172 and 16 <= second <= 31:
        return "172_private"

    return "other"


def get_last_octet(ip):
    return int(ip.split(".")[-1])


def has_used_conflict(network_name, network):
    for used in USED_SUBNETS_BY_NETWORK[network_name]:
        if network.overlaps(used):
            return True

    return False


def register_subnet(network_name, network):
    USED_SUBNETS_BY_NETWORK[network_name].append(network)


def random_host_block(prefix):
    block_size = 2 ** (32 - prefix)

    if block_size >= 256:
        return 0

    return random.choice(list(range(0, 256, block_size)))


def make_candidate_network(base, prefix):
    a = random.randint(1, 250)
    b = random.randint(1, 250)
    block = random_host_block(prefix)

    if base == "10":
        if prefix <= 24:
            return ipaddress.ip_network(f"10.{a}.{b}.0/{prefix}", strict=False)

        return ipaddress.ip_network(f"10.{a}.{b}.{block}/{prefix}", strict=False)

    if base == "192.168":
        if prefix <= 24:
            return ipaddress.ip_network(f"192.168.{a}.0/{prefix}", strict=False)

        return ipaddress.ip_network(f"192.168.{a}.{block}/{prefix}", strict=False)

    if base == "172.16":
        second_octet = random.randint(16, 31)

        if prefix <= 24:
            return ipaddress.ip_network(f"172.{second_octet}.{a}.0/{prefix}", strict=False)

        return ipaddress.ip_network(f"172.{second_octet}.{a}.{block}/{prefix}", strict=False)

    raise ValueError("Unknown IP base")


def create_network_for_profile(
    network_name,
    profile_name,
    prefix,
    force_wrong_range=False,
    register=True,
    avoid_conflict=True
):
    profile = PROFILES[profile_name]

    if force_wrong_range:
        possible_bases = ["10", "192.168", "172.16"]
        expected = profile["expected_ranges"]

        possible_bases = [
            base for base in possible_bases
            if base not in expected
        ]

        if prefix <= 23 and "192.168" in possible_bases and len(possible_bases) > 1:
            possible_bases = [
                base for base in possible_bases
                if base != "192.168"
            ]

        base = random.choice(possible_bases)

    else:
        base = profile["normal_ip_base"]

    for _ in range(50000):
        network = make_candidate_network(base, prefix)

        if avoid_conflict and has_used_conflict(network_name, network):
            continue

        hosts = list(network.hosts())

        if len(hosts) < 2:
            continue

        ip_a = str(hosts[0])
        ip_b = str(hosts[1])

        if register:
            register_subnet(network_name, network)

        return network, ip_a, ip_b

    raise RuntimeError(
        f"Could not create non-overlapping subnet for {network_name} {profile_name}/{prefix}"
    )


def choose_device_pair(network_name, role_a, role_b):
    if role_a == role_b:
        choices = random.sample(DEVICE_POOLS[role_a], 2)

        device_a = f"{network_name}_{choices[0]}"
        device_b = f"{network_name}_{choices[1]}"

    else:
        device_a = f"{network_name}_{random.choice(DEVICE_POOLS[role_a])}"
        device_b = f"{network_name}_{random.choice(DEVICE_POOLS[role_b])}"

    return device_a, device_b


def build_row(
    sample_id,
    network_name,
    profile_name,
    ip_a,
    ip_b,
    prefix_a,
    prefix_b,
    role_a,
    role_b,
    label,
    risk_level,
    error_type
):
    device_a, device_b = choose_device_pair(network_name, role_a, role_b)

    return {
        "Sample_ID": sample_id,
        "Network": network_name,
        "Link_ID": f"L{sample_id}",
        "Link_Profile": profile_name,
        "Device_A": device_a,
        "Device_B": device_b,
        "Device_A_Role": role_a,
        "Device_B_Role": role_b,
        "Interface_A": "Gi0/1",
        "Interface_B": "Gi0/1",
        "IP_A": ip_a,
        "IP_B": ip_b,
        "Subnet_A": prefix_to_mask(prefix_a),
        "Subnet_B": prefix_to_mask(prefix_b),
        "Prefix_A": prefix_a,
        "Prefix_B": prefix_b,
        "Status_A": "up",
        "Status_B": "up",
        "Label": label,
        "Risk_Level": risk_level,
        "Error_Type": error_type
    }


# =====================================================
# 3. NORMAL DATA GENERATION
# =====================================================

def create_normal_sample(sample_id, profile_name):
    network_name = random.choice(NETWORKS)

    profile = PROFILES[profile_name]

    prefix = random.choice(profile["normal_prefixes"])

    network, ip_a, ip_b = create_network_for_profile(
        network_name,
        profile_name,
        prefix
    )

    role_a, role_b = random.choice(profile["normal_roles"])

    return build_row(
        sample_id,
        network_name,
        profile_name,
        ip_a,
        ip_b,
        prefix,
        prefix,
        role_a,
        role_b,
        label=0,
        risk_level="Low",
        error_type="none"
    )


# =====================================================
# 4. DIRECT ERROR GENERATION
# =====================================================

def create_direct_error_sample(sample_id, error_type):
    profile_name = random.choice(list(PROFILES.keys()))

    row = create_normal_sample(sample_id, profile_name)

    row["Label"] = 1
    row["Risk_Level"] = "High"
    row["Error_Type"] = error_type

    if error_type == "duplicate_ip":
        row["IP_B"] = row["IP_A"]

    elif error_type == "subnet_mask_mismatch":
        possible_prefixes = [23, 24, 25, 28, 29, 30]

        wrong_prefix = random.choice([
            prefix for prefix in possible_prefixes
            if prefix != row["Prefix_A"]
        ])

        row["Prefix_B"] = wrong_prefix
        row["Subnet_B"] = prefix_to_mask(wrong_prefix)
        row["Risk_Level"] = "Medium"

    elif error_type == "network_mismatch":
        wrong_network, wrong_ip_a, wrong_ip_b = create_network_for_profile(
            row["Network"],
            row["Link_Profile"],
            row["Prefix_B"],
            force_wrong_range=True,
            register=False,
            avoid_conflict=True
        )

        row["IP_B"] = wrong_ip_b

    elif error_type == "status_mismatch":
        row["Status_B"] = "down"
        row["Risk_Level"] = "Medium"

    elif error_type == "invalid_host_ip":
        network = ipaddress.ip_network(
            get_network_cidr(row["IP_A"], row["Subnet_A"]),
            strict=False
        )

        row["IP_A"] = str(network.network_address)

    return row


# =====================================================
# 5. CONTEXTUAL ANOMALY GENERATION
# =====================================================

def create_contextual_anomaly_sample(sample_id, anomaly_type):
    network_name = random.choice(NETWORKS)

    if anomaly_type == "profile_subnet_mismatch":
        profile_name = random.choice(list(PROFILES.keys()))

        if profile_name == "router_to_router":
            wrong_prefix = 24

        elif profile_name == "branch_lan":
            wrong_prefix = 30

        elif profile_name == "campus_network":
            wrong_prefix = 30

        else:
            wrong_prefix = 24

        network, ip_a, ip_b = create_network_for_profile(
            network_name,
            profile_name,
            wrong_prefix
        )

        role_a, role_b = random.choice(PROFILES[profile_name]["normal_roles"])

        return build_row(
            sample_id,
            network_name,
            profile_name,
            ip_a,
            ip_b,
            wrong_prefix,
            wrong_prefix,
            role_a,
            role_b,
            label=1,
            risk_level="Medium",
            error_type=anomaly_type
        )

    elif anomaly_type == "device_role_mismatch":
        profile_name = random.choice(list(PROFILES.keys()))

        prefix = random.choice(PROFILES[profile_name]["normal_prefixes"])

        network, ip_a, ip_b = create_network_for_profile(
            network_name,
            profile_name,
            prefix
        )

        wrong_roles = {
            "router_to_router": ("PC", "PC"),
            "branch_lan": ("Server", "Firewall"),
            "campus_network": ("PC", "Server"),
            "datacenter_segment": ("PC", "AccessPoint")
        }

        role_a, role_b = wrong_roles[profile_name]

        return build_row(
            sample_id,
            network_name,
            profile_name,
            ip_a,
            ip_b,
            prefix,
            prefix,
            role_a,
            role_b,
            label=1,
            risk_level="Medium",
            error_type=anomaly_type
        )

    elif anomaly_type == "ip_range_profile_mismatch":
        profile_name = random.choice(list(PROFILES.keys()))

        prefix = random.choice(PROFILES[profile_name]["normal_prefixes"])

        network, ip_a, ip_b = create_network_for_profile(
            network_name,
            profile_name,
            prefix,
            force_wrong_range=True
        )

        role_a, role_b = random.choice(PROFILES[profile_name]["normal_roles"])

        return build_row(
            sample_id,
            network_name,
            profile_name,
            ip_a,
            ip_b,
            prefix,
            prefix,
            role_a,
            role_b,
            label=1,
            risk_level="Medium",
            error_type=anomaly_type
        )

    elif anomaly_type == "gateway_role_mismatch":
        profile_name = random.choice(["branch_lan", "campus_network"])

        prefix = 24

        network, ip_a, ip_b = create_network_for_profile(
            network_name,
            profile_name,
            prefix
        )

        return build_row(
            sample_id,
            network_name,
            profile_name,
            ip_a,
            ip_b,
            prefix,
            prefix,
            "PC",
            "Router",
            label=1,
            risk_level="Medium",
            error_type=anomaly_type
        )

    raise ValueError("Unknown contextual anomaly type")


# =====================================================
# 6. CROSS-LINK ANOMALY GENERATION
# =====================================================

def create_duplicate_subnet_pair(sample_id):
    network_name = random.choice(NETWORKS)

    profile_name = "datacenter_segment"
    prefix = 29

    network, ip_a, ip_b = create_network_for_profile(
        network_name,
        profile_name,
        prefix
    )

    hosts = list(network.hosts())

    row1 = build_row(
        sample_id,
        network_name,
        profile_name,
        str(hosts[0]),
        str(hosts[1]),
        prefix,
        prefix,
        "Switch",
        "Server",
        label=1,
        risk_level="High",
        error_type="duplicate_subnet"
    )

    row2 = build_row(
        sample_id + 1,
        network_name,
        profile_name,
        str(hosts[2]),
        str(hosts[3]),
        prefix,
        prefix,
        "Firewall",
        "Server",
        label=1,
        risk_level="High",
        error_type="duplicate_subnet"
    )

    return row1, row2


def create_overlapping_subnet_pair(sample_id):
    network_name = random.choice(NETWORKS)

    for _ in range(50000):
        a = random.randint(1, 250)
        b = random.randint(1, 250)

        parent_network = ipaddress.ip_network(
            f"10.{a}.{b}.0/28",
            strict=False
        )

        child_network = ipaddress.ip_network(
            f"10.{a}.{b}.0/30",
            strict=False
        )

        if has_used_conflict(network_name, parent_network) or has_used_conflict(network_name, child_network):
            continue

        register_subnet(network_name, parent_network)
        register_subnet(network_name, child_network)

        parent_hosts = list(parent_network.hosts())
        child_hosts = list(child_network.hosts())

        row1 = build_row(
            sample_id,
            network_name,
            "datacenter_segment",
            str(parent_hosts[4]),
            str(parent_hosts[5]),
            28,
            28,
            "Switch",
            "Server",
            label=1,
            risk_level="High",
            error_type="overlapping_subnet"
        )

        row2 = build_row(
            sample_id + 1,
            network_name,
            "router_to_router",
            str(child_hosts[0]),
            str(child_hosts[1]),
            30,
            30,
            "Router",
            "Router",
            label=1,
            risk_level="High",
            error_type="overlapping_subnet"
        )

        return row1, row2

    raise RuntimeError("Could not create overlapping subnet pair")


# =====================================================
# 7. GRAPH-BASED FEATURE EXTRACTION
# =====================================================

def add_graph_based_features(raw_df):
    raw_df = raw_df.copy()

    # Device degree feature
    device_degree = defaultdict(int)

    for _, row in raw_df.iterrows():
        device_degree[row["Device_A"]] += 1
        device_degree[row["Device_B"]] += 1

    raw_df["device_a_degree"] = raw_df["Device_A"].map(device_degree)
    raw_df["device_b_degree"] = raw_df["Device_B"].map(device_degree)

    # Subnet CIDR
    raw_df["Subnet_CIDR_A"] = raw_df.apply(
        lambda row: get_network_cidr(row["IP_A"], row["Subnet_A"]),
        axis=1
    )

    # Duplicate subnet feature inside the same network
    raw_df["subnet_reuse_count"] = raw_df.groupby(
        ["Network", "Subnet_CIDR_A"]
    )["Subnet_CIDR_A"].transform("count")

    raw_df["duplicate_subnet_flag"] = (
        raw_df["subnet_reuse_count"] > 1
    ).astype(int)

    # Overlapping subnet feature inside the same network
    raw_df["overlapping_subnet_flag"] = 0

    for network_name, group in raw_df.groupby("Network"):
        idx_list = group.index.tolist()

        subnet_objects = [
            ipaddress.ip_network(raw_df.loc[idx, "Subnet_CIDR_A"], strict=False)
            for idx in idx_list
        ]

        for local_i, subnet_i in enumerate(subnet_objects):
            overlap_found = 0

            for local_j, subnet_j in enumerate(subnet_objects):
                if local_i == local_j:
                    continue

                if subnet_i.overlaps(subnet_j) and subnet_i != subnet_j:
                    overlap_found = 1
                    break

            raw_df.loc[idx_list[local_i], "overlapping_subnet_flag"] = overlap_found

    # Role pair validity
    valid_role_pairs = {
        "router_to_router": [("Router", "Router")],
        "branch_lan": [("Router", "PC"), ("Router", "Switch")],
        "campus_network": [("Router", "Switch"), ("Router", "AccessPoint")],
        "datacenter_segment": [("Firewall", "Server"), ("Switch", "Server")]
    }

    def check_role_pair_valid(row):
        profile = row["Link_Profile"]

        role_pair = (
            row["Device_A_Role"],
            row["Device_B_Role"]
        )

        reverse_role_pair = (
            row["Device_B_Role"],
            row["Device_A_Role"]
        )

        allowed_pairs = valid_role_pairs.get(profile, [])

        if role_pair in allowed_pairs or reverse_role_pair in allowed_pairs:
            return 1

        return 0

    raw_df["role_pair_valid"] = raw_df.apply(check_role_pair_valid, axis=1)

    return raw_df


# =====================================================
# 8. FEATURE DATASET GENERATION
# =====================================================

def generate_features(raw_df):
    feature_rows = []

    raw_df = add_graph_based_features(raw_df)

    for _, row in raw_df.iterrows():
        ip_a = row["IP_A"]
        ip_b = row["IP_B"]

        subnet_a = row["Subnet_A"]
        subnet_b = row["Subnet_B"]

        prefix_a = get_prefix(subnet_a)
        prefix_b = get_prefix(subnet_b)

        duplicate_ip = int(ip_a == ip_b)

        same_subnet_mask = int(subnet_a == subnet_b)

        same_network = int(
            get_network_address(ip_a, subnet_a)
            ==
            get_network_address(ip_b, subnet_b)
        )

        prefix_difference = abs(prefix_a - prefix_b)

        both_interfaces_up = int(
            row["Status_A"] == "up"
            and
            row["Status_B"] == "up"
        )

        invalid_host_a = int(
            not is_valid_host_ip(ip_a, subnet_a)
        )

        invalid_host_b = int(
            not is_valid_host_ip(ip_b, subnet_b)
        )

        subnet_capacity_a = subnet_capacity(prefix_a)
        subnet_capacity_b = subnet_capacity(prefix_b)

        host_number_a = get_last_octet(ip_a)
        host_number_b = get_last_octet(ip_b)

        gateway_ip_on_non_router = int(
            row["Link_Profile"] in ["branch_lan", "campus_network"]
            and
            host_number_a == 1
            and
            row["Device_A_Role"] != "Router"
        )

        ip_range_a = get_ip_range_type(ip_a)
        ip_range_b = get_ip_range_type(ip_b)

        expected_ranges = PROFILES[row["Link_Profile"]]["expected_ranges"]

        expected_ip_range_match = int(
            (
                ip_range_a == "10_private"
                and
                "10" in expected_ranges
            )
            or
            (
                ip_range_a == "192_private"
                and
                "192.168" in expected_ranges
            )
            or
            (
                ip_range_a == "172_private"
                and
                "172.16" in expected_ranges
            )
        )

        feature_rows.append({
            "Sample_ID": row["Sample_ID"],
            "Network": row["Network"],
            "Link_ID": row["Link_ID"],
            "Link_Profile": row["Link_Profile"],
            "Device_A_Role": row["Device_A_Role"],
            "Device_B_Role": row["Device_B_Role"],

            # Basic IP/subnet features
            "Prefix_A": prefix_a,
            "Prefix_B": prefix_b,
            "duplicate_ip": duplicate_ip,
            "same_subnet_mask": same_subnet_mask,
            "same_network": same_network,
            "prefix_difference": prefix_difference,
            "both_interfaces_up": both_interfaces_up,
            "invalid_host_a": invalid_host_a,
            "invalid_host_b": invalid_host_b,
            "subnet_capacity_a": subnet_capacity_a,
            "subnet_capacity_b": subnet_capacity_b,
            "host_number_a": host_number_a,
            "host_number_b": host_number_b,
            "gateway_ip_on_non_router": gateway_ip_on_non_router,
            "expected_ip_range_match": expected_ip_range_match,

            # Graph-based features
            "device_a_degree": row["device_a_degree"],
            "device_b_degree": row["device_b_degree"],
            "subnet_reuse_count": row["subnet_reuse_count"],
            "duplicate_subnet_flag": row["duplicate_subnet_flag"],
            "overlapping_subnet_flag": row["overlapping_subnet_flag"],
            "role_pair_valid": row["role_pair_valid"],

            # Categorical features
            "IP_Range_A": ip_range_a,
            "IP_Range_B": ip_range_b,

            # Labels
            "Label": row["Label"],
            "Risk_Level": row["Risk_Level"],
            "Error_Type": row["Error_Type"]
        })

    return pd.DataFrame(feature_rows)


# =====================================================
# 9. GENERATE FULL DATASET
# =====================================================

rows = []
sample_id = 1

# Normal data
for profile_name in PROFILES.keys():
    for _ in range(NORMAL_PER_PROFILE):
        rows.append(
            create_normal_sample(sample_id, profile_name)
        )

        sample_id += 1

# Direct errors
for error_type in DIRECT_ERRORS:
    for _ in range(DIRECT_ERROR_PER_TYPE):
        rows.append(
            create_direct_error_sample(sample_id, error_type)
        )

        sample_id += 1

# Contextual anomalies
for anomaly_type in CONTEXTUAL_ANOMALIES:
    for _ in range(CONTEXTUAL_ANOMALY_PER_TYPE):
        rows.append(
            create_contextual_anomaly_sample(sample_id, anomaly_type)
        )

        sample_id += 1

# Duplicate subnet anomalies
for _ in range(CROSS_LINK_ANOMALY_PER_TYPE):
    row1, row2 = create_duplicate_subnet_pair(sample_id)

    rows.append(row1)
    rows.append(row2)

    sample_id += 2

# Overlapping subnet anomalies
for _ in range(CROSS_LINK_ANOMALY_PER_TYPE):
    row1, row2 = create_overlapping_subnet_pair(sample_id)

    rows.append(row1)
    rows.append(row2)

    sample_id += 2

# Create raw dataframe
raw_df = pd.DataFrame(rows)

raw_df = raw_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# Create feature dataframe
features_df = generate_features(raw_df)


# =====================================================
# 10. SAVE DATASET FILES
# =====================================================

raw_df.to_csv("dataset_v5_raw.csv", index=False)

features_df.to_csv("dataset_v5_features.csv", index=False)

with pd.ExcelWriter("dataset_v5_final.xlsx", engine="openpyxl") as writer:
    raw_df.to_excel(
        writer,
        sheet_name="Raw_Dataset",
        index=False
    )

    features_df.to_excel(
        writer,
        sheet_name="Feature_Dataset",
        index=False
    )


# =====================================================
# 11. BASIC CHECKS
# =====================================================

print("Dataset created successfully.")

print("\nRaw dataset shape:")
print(raw_df.shape)

print("\nFeature dataset shape:")
print(features_df.shape)

print("\nLabel distribution:")
print(raw_df["Label"].value_counts())

print("\nRisk level distribution:")
print(raw_df["Risk_Level"].value_counts())

print("\nError type distribution:")
print(raw_df["Error_Type"].value_counts())

print("\nLink profile distribution:")
print(raw_df["Link_Profile"].value_counts())

print("\nMissing values in raw dataset:")
print(raw_df.isnull().sum().sum())

print("\nMissing values in feature dataset:")
print(features_df.isnull().sum().sum())

print("\nGraph-based feature check by error type:")
print(
    features_df.groupby("Error_Type")[
        [
            "duplicate_subnet_flag",
            "overlapping_subnet_flag",
            "role_pair_valid",
            "expected_ip_range_match",
            "gateway_ip_on_non_router"
        ]
    ].mean()
)

print("\nFiles saved:")
print("dataset_v5_raw.csv")
print("dataset_v5_features.csv")
print("dataset_v5_final.xlsx")

Dataset created successfully.

Raw dataset shape:
(5000, 21)

Feature dataset shape:
(5000, 32)

Label distribution:
Label
1    2600
0    2400
Name: count, dtype: int64

Risk level distribution:
Risk_Level
Low       2400
Medium    1480
High      1120
Name: count, dtype: int64

Error type distribution:
Error_Type
none                         2400
profile_subnet_mismatch       250
gateway_role_mismatch         250
device_role_mismatch          250
ip_range_profile_mismatch     250
network_mismatch              240
duplicate_ip                  240
subnet_mask_mismatch          240
invalid_host_ip               240
status_mismatch               240
duplicate_subnet              200
overlapping_subnet            200
Name: count, dtype: int64

Link profile distribution:
Link_Profile
datacenter_segment    1414
branch_lan            1220
campus_network        1188
router_to_router      1178
Name: count, dtype: int64

Missing values in raw dataset:
0

Missing values in feature dataset:
0

Grap

In [3]:
import pandas as pd
import ipaddress

# =====================================================
# 1. LOAD RAW DATASET
# =====================================================

df = pd.read_csv("dataset_v5_raw.csv")


# =====================================================
# 2. HELPER FUNCTIONS
# =====================================================

def get_network_address(ip, mask):
    return str(ipaddress.ip_network(f"{ip}/{mask}", strict=False).network_address)


def is_valid_host_ip(ip, mask):
    network = ipaddress.ip_network(f"{ip}/{mask}", strict=False)
    ip_obj = ipaddress.ip_address(ip)

    if ip_obj == network.network_address:
        return False

    if ip_obj == network.broadcast_address:
        return False

    return True


# =====================================================
# 3. BASIC RULE-BASED BASELINE MODEL
#    Only five direct IP configuration rules are used.
# =====================================================

def basic_rule_based_validate(row):
    ip_a = row["IP_A"]
    ip_b = row["IP_B"]

    subnet_a = row["Subnet_A"]
    subnet_b = row["Subnet_B"]

    status_a = row["Status_A"]
    status_b = row["Status_B"]

    # Rule 1: Duplicate IP address
    if ip_a == ip_b:
        return 1, "duplicate_ip", "High"

    # Rule 2: Invalid host IP address
    if not is_valid_host_ip(ip_a, subnet_a) or not is_valid_host_ip(ip_b, subnet_b):
        return 1, "invalid_host_ip", "High"

    # Rule 3: Subnet mask mismatch
    if subnet_a != subnet_b:
        return 1, "subnet_mask_mismatch", "Medium"

    # Rule 4: Network mismatch
    if get_network_address(ip_a, subnet_a) != get_network_address(ip_b, subnet_b):
        return 1, "network_mismatch", "High"

    # Rule 5: Interface status mismatch
    if status_a != status_b:
        return 1, "status_mismatch", "Medium"

    # If none of the five direct rules are violated
    return 0, "none", "Low"


# =====================================================
# 4. APPLY RULE-BASED MODEL
# =====================================================

results = df.apply(basic_rule_based_validate, axis=1)

df["Basic_Rule_Label"] = [result[0] for result in results]
df["Basic_Rule_Error_Type"] = [result[1] for result in results]
df["Basic_Rule_Risk_Level"] = [result[2] for result in results]


# =====================================================
# 5. BASIC EVALUATION
# =====================================================

label_accuracy = (df["Label"] == df["Basic_Rule_Label"]).mean()

print("Basic Rule-Based Validation Completed")
print("Total Records:", len(df))

print("\nOverall Label Accuracy:")
print(label_accuracy)

print("\nActual Label Distribution:")
print(df["Label"].value_counts())

print("\nBasic Rule-Based Prediction Distribution:")
print(df["Basic_Rule_Label"].value_counts())

print("\nConfusion Matrix:")
print(
    pd.crosstab(
        df["Label"],
        df["Basic_Rule_Label"],
        rownames=["Actual"],
        colnames=["Predicted"]
    )
)


# =====================================================
# 6. DIRECT ERROR PERFORMANCE
# =====================================================

direct_error_types = [
    "duplicate_ip",
    "subnet_mask_mismatch",
    "network_mismatch",
    "status_mismatch",
    "invalid_host_ip"
]

direct_error_df = df[df["Error_Type"].isin(direct_error_types)]

direct_error_accuracy = (
    direct_error_df["Error_Type"] == direct_error_df["Basic_Rule_Error_Type"]
).mean()

print("\nDirect Error Detection Accuracy:")
print(direct_error_accuracy)

print("\nDirect Error Actual Distribution:")
print(direct_error_df["Error_Type"].value_counts())

print("\nDirect Error Predicted Distribution:")
print(direct_error_df["Basic_Rule_Error_Type"].value_counts())


# =====================================================
# 7. MISSED ANOMALIES
#    These are expected to be missed by the basic rule model.
# =====================================================

missed_anomalies = df[
    (df["Label"] == 1) &
    (df["Basic_Rule_Label"] == 0)
]

print("\nMissed Anomalies by Basic Rule-Based Model:")
print(missed_anomalies["Error_Type"].value_counts())

print("\nTotal Missed Anomalies:")
print(len(missed_anomalies))


# =====================================================
# 8. SAVE RESULTS
# =====================================================

df.to_csv("basic_rule_based_v5_results.csv", index=False)

with pd.ExcelWriter("basic_rule_based_v5_results.xlsx", engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Basic_Rule_Results", index=False)

print("\nFiles saved:")
print("basic_rule_based_v5_results.csv")
print("basic_rule_based_v5_results.xlsx")

Basic Rule-Based Validation Completed
Total Records: 5000

Overall Label Accuracy:
0.72

Actual Label Distribution:
Label
1    2600
0    2400
Name: count, dtype: int64

Basic Rule-Based Prediction Distribution:
Basic_Rule_Label
0    3800
1    1200
Name: count, dtype: int64

Confusion Matrix:
Predicted     0     1
Actual               
0          2400     0
1          1400  1200

Direct Error Detection Accuracy:
1.0

Direct Error Actual Distribution:
Error_Type
duplicate_ip            240
subnet_mask_mismatch    240
network_mismatch        240
invalid_host_ip         240
status_mismatch         240
Name: count, dtype: int64

Direct Error Predicted Distribution:
Basic_Rule_Error_Type
duplicate_ip            240
subnet_mask_mismatch    240
network_mismatch        240
invalid_host_ip         240
status_mismatch         240
Name: count, dtype: int64

Missed Anomalies by Basic Rule-Based Model:
Error_Type
device_role_mismatch         250
ip_range_profile_mismatch    250
gateway_role_mismatch